## 📦 Cell 1: Install Dependencies

In [ ]:
# Install semua dependency yang dibutuhkan
!pip install -q datasets transformers torch
!pip install -q torch-geometric
!pip install -q scikit-learn seaborn matplotlib imbalanced-learn

print("✅ Semua dependency berhasil diinstall!")

## 📚 Cell 2: Import Library

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Cek GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Menggunakan device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 📥 Cell 3: Load & Sample Dataset

In [ ]:
# ── Konfigurasi ──────────────────────────────────────────────
SAMPLE_SIZE   = 8000   # total email yang dipakai (ubah sesuai kebutuhan)
RANDOM_SEED   = 42
# ─────────────────────────────────────────────────────────────

print("⏳ Loading dataset...")
# dataset = load_dataset("indonesian_phishing_dataset.xlsx", split="train")
df = pd.read_excel("indonesian_phishing_dataset.xlsx")

print(f"✅ Dataset loaded: {len(df):,} baris")
print(f"   Kolom: {list(df.columns)}")
print(f"\nDistribusi label (full):")
print(df['label'].value_counts())

In [ ]:
# Sampling stratified (jaga rasio spam:ham)
spam_df = df[df['label'] == 1]
ham_df  = df[df['label'] == 0]

n_spam = min(SAMPLE_SIZE // 2, len(spam_df))
n_ham  = min(SAMPLE_SIZE // 2, len(ham_df))

df = pd.concat([
    spam_df.sample(n=n_spam, random_state=RANDOM_SEED),
    ham_df.sample(n=n_ham,  random_state=RANDOM_SEED)
]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

df['email_id'] = [f"E{str(i).zfill(4)}" for i in range(len(df))]

print(f"✅ Dataset setelah sampling: {len(df):,} baris")
print(f"\nDistribusi label (sampled):")
print(df['label'].value_counts())
print(f"\nPreview:")
df[['email_id', 'sender', 'subject_id', 'label']].head(10)

## 🧹 Cell 4: Preprocessing

In [ ]:
def extract_domain(email_addr):
    """Ekstrak domain dari alamat email."""
    match = re.search(r"@([\w.-]+)", str(email_addr))
    return match.group(1) if match else "unknown"

def preprocess_text(text):
    """Bersihkan teks email."""
    if pd.isna(text) or text is None:
        return ""
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "[URL]", text)   # ganti URL
    text = re.sub(r"\S+@\S+", "[EMAIL]", text)         # ganti email address
    text = re.sub(r"[^\w\s\[\].,!?]", " ", text)       # hapus karakter aneh
    text = re.sub(r"\s+", " ", text).strip()            # normalisasi whitespace
    return text[:512]                                    # batasi panjang

# Gabungkan subject + body sebagai input teks
# df['subject_clean'] = df['subject_id'].apply(preprocess_text)
df['body_clean']    = df['text_id'].apply(preprocess_text)
df['full_text']     = df['body_clean']
df['sender_domain'] = df['sender'].apply(extract_domain)

# Handle missing sender
df['sender'] = df['sender'].fillna('unknown@unknown.com')

print(f"✅ Preprocessing selesai")
print(f"\nContoh teks setelah preprocessing:")
print(df['full_text'].iloc[0][:300])

## 🤖 Cell 5: Load BERT & Ekstrak Node Features

> ⚠️ Cell ini bisa memakan waktu 5–15 menit tergantung jumlah data dan GPU.

In [ ]:
# ── Konfigurasi BERT ──────────────────────────────────────────
# Karena dataset ini berbahasa Inggris, pakai bert-base-uncased
# Jika dataset Anda berbahasa Indonesia, ganti dengan:
BERT_MODEL = "indobenchmark/indobert-base-p1"
# BERT_MODEL  = "bert-base-uncased"
MAX_LENGTH  = 128   # panjang token maksimal
BATCH_SIZE  = 16    # batch size untuk embedding extraction
# ─────────────────────────────────────────────────────────────

print(f"⏳ Loading {BERT_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()
print(f"✅ BERT loaded!")

In [ ]:
def extract_embeddings_batch(texts, batch_size=BATCH_SIZE):
    """Ekstrak [CLS] embedding dari BERT secara batch."""
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_num   = i // batch_size + 1

        if batch_num % 10 == 0 or batch_num == 1:
            print(f"  Batch {batch_num}/{total_batches}...", end="\r")

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True
        ).to(device)

        with torch.no_grad():
            output = bert_model(**inputs)

        # Ambil [CLS] token embedding
        cls_embeddings = output.last_hidden_state[:, 0, :].cpu()
        all_embeddings.append(cls_embeddings)

    return torch.cat(all_embeddings, dim=0)  # (N, 768)

EMBED_CACHE = "node_features.pt"

if os.path.exists(EMBED_CACHE):
    print("⚡ Load embedding dari cache...")
    node_features = torch.load(EMBED_CACHE)
else:
    print("⏳ Mengekstrak embeddings...")
    texts = df['full_text'].tolist()
    node_features = extract_embeddings_batch(texts)
    torch.save(node_features, EMBED_CACHE)
    print("\n✅ Embeddings disimpan ke cache!")

print(f"\n✅ Node features shape: {node_features.shape}")
# Expected: (N, 768)

## 🕸️ Cell 6: Bangun Graf

In [ ]:
def build_sender_graph(df, min_group_size=2):
    """
    Bangun edge antar email dari sender yang sama.
    Email dari sender yang sama saling terhubung (fully connected per group).
    """
    edge_list = []
    sender_groups = df.groupby('sender').indices

    n_groups_with_edges = 0
    for sender, indices in sender_groups.items():
        if len(indices) < min_group_size:
            continue
        n_groups_with_edges += 1
        indices = list(indices)
        for i in range(len(indices)):
            for j in range(len(indices)):
                if i != j:
                    edge_list.append([indices[i], indices[j]])

    print(f"   Jumlah sender unik     : {len(sender_groups):,}")
    print(f"   Sender dengan >1 email : {n_groups_with_edges:,}")
    print(f"   Total edges            : {len(edge_list):,}")

    if len(edge_list) == 0:
        print("⚠️  Tidak ada edge! Menggunakan self-loop sebagai fallback.")
        edge_list = [[i, i] for i in range(len(df))]

    edge_index = torch.tensor(edge_list, dtype=torch.long).T  # (2, E)
    return edge_index

print("⏳ Membangun graf...")
edge_index = build_sender_graph(df)
print(f"\n✅ Graf selesai dibangun!")
print(f"   Edge index shape: {edge_index.shape}")

In [ ]:
# Buat PyG Data object + train/val/test mask
labels = torch.tensor(df['label'].values, dtype=torch.long)
N = len(df)
indices = np.arange(N)

train_idx, test_idx = train_test_split(
    indices, test_size=0.2,
    stratify=df['label'].values,
    random_state=RANDOM_SEED
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.125,  # 0.125 x 0.8 = 0.1 dari total
    stratify=df.iloc[train_idx]['label'].values,
    random_state=RANDOM_SEED
)

data = Data(
    x          = node_features,
    edge_index = edge_index,
    y          = labels
)

data.train_mask = torch.zeros(N, dtype=torch.bool)
data.val_mask   = torch.zeros(N, dtype=torch.bool)
data.test_mask  = torch.zeros(N, dtype=torch.bool)

data.train_mask[train_idx] = True
data.val_mask[val_idx]     = True
data.test_mask[test_idx]   = True

data = data.to(device)

print(f"✅ PyG Data object siap!")
print(f"   Train: {data.train_mask.sum().item():,} nodes")
print(f"   Val  : {data.val_mask.sum().item():,} nodes")
print(f"   Test : {data.test_mask.sum().item():,} nodes")
print(f"\n{data}")

## 🧠 Cell 7: Definisi Model SpamGAT

In [ ]:
class SpamGAT(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=128, num_classes=2, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        # Proyeksikan dari BERT dim (768) ke hidden dim
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # GAT Layer 1: hidden_dim → hidden_dim * heads
        self.gat1 = GATConv(
            hidden_dim,
            hidden_dim,
            heads=heads,
            dropout=dropout,
            concat=True
        )

        # GAT Layer 2: hidden_dim * heads → hidden_dim
        self.gat2 = GATConv(
            hidden_dim * heads,
            hidden_dim,
            heads=1,
            dropout=dropout,
            concat=False
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, return_attention=False):
        # 1. Proyeksi input
        x = self.input_proj(x)

        # 2. GAT layer 1
        if return_attention:
            x, (ei1, alpha1) = self.gat1(x, edge_index, return_attention_weights=True)
        else:
            x = self.gat1(x, edge_index)

        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # 3. GAT layer 2
        if return_attention:
            x, (ei2, alpha2) = self.gat2(x, edge_index, return_attention_weights=True)
        else:
            x = self.gat2(x, edge_index)

        x = F.elu(x)

        # 4. Klasifikasi
        out = self.classifier(x)

        if return_attention:
            return out, (ei1, alpha1), (ei2, alpha2)
        return out


model = SpamGAT(
    in_dim=768,
    hidden_dim=128,
    num_classes=2,
    heads=4,
    dropout=0.3
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model SpamGAT siap!")
print(f"   Total trainable parameters: {total_params:,}")
print(model)

## 🏋️ Cell 8: Training

In [ ]:
# ── Konfigurasi Training ──────────────────────────────────────
EPOCHS    = 100
LR        = 1e-3
WD        = 1e-4   # weight decay
PATIENCE  = 15     # early stopping patience
# ─────────────────────────────────────────────────────────────

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

# Hitung class weight untuk handle imbalance
n_spam = (data.y[data.train_mask] == 1).sum().float()
n_ham  = (data.y[data.train_mask] == 0).sum().float()
class_weight = torch.tensor([n_spam / (n_spam + n_ham), n_ham / (n_spam + n_ham)]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weight)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0

print(f"⏳ Mulai training ({EPOCHS} epochs)...\n")

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # ── Validasi ──
    model.eval()
    with torch.no_grad():
        val_out  = model(data.x, data.edge_index)
        val_loss = criterion(val_out[data.val_mask], data.y[data.val_mask]).item()
        val_pred = val_out[data.val_mask].argmax(dim=1)
        val_acc  = (val_pred == data.y[data.val_mask]).float().mean().item()

    scheduler.step(val_acc)
    history['train_loss'].append(loss.item())
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Log setiap 10 epoch
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"Train Loss: {loss.item():.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")

    # Early stopping + simpan best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹️  Early stopping di epoch {epoch}")
            break

print(f"\n✅ Training selesai! Best Val Acc: {best_val_acc:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', color='#4E79A7')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='#F28E2B')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(history['val_acc'], label='Val Accuracy', color='#59A14F')
axes[1].axhline(y=best_val_acc, color='red', linestyle='--', alpha=0.5, label=f'Best: {best_val_acc:.4f}')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Cell 9: Evaluasi Model

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

with torch.no_grad():
    out  = model(data.x, data.edge_index)
    pred = out[data.test_mask].argmax(dim=1).cpu().numpy()
    true = data.y[data.test_mask].cpu().numpy()
    probs = F.softmax(out[data.test_mask], dim=1).cpu().numpy()

print("=" * 50)
print("📊 CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(true, pred, target_names=['Ham', 'Spam']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(true, pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Ham', 'Spam'],
    yticklabels=['Ham', 'Spam'],
    linewidths=0.5
)
plt.title('Confusion Matrix - SpamGAT', fontsize=13, pad=12)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Cell 10: Visualisasi Attention Weight GAT

In [ ]:
# Ekstrak attention weights dari model
model.eval()
with torch.no_grad():
    out, (ei1, alpha1), (ei2, alpha2) = model(
        data.x, data.edge_index, return_attention=True
    )

print(f"✅ Attention weights diekstrak")
print(f"   Layer 1 alpha shape: {alpha1.shape}")  # (E, heads)
print(f"   Layer 2 alpha shape: {alpha2.shape}")  # (E, 1)

In [ ]:
# Visualisasi attention heatmap per node
from torch_geometric.utils import add_self_loops

edge_index_with_loops, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

def visualize_node_attention(df, edge_index, alpha, node_id, layer=1, top_k=10):
    avg_alpha = alpha.cpu().mean(dim=1).numpy()
    ei        = edge_index.cpu().numpy()  # pakai edge_index yang sudah ada self-loops

    mask    = ei[1] == node_id
    sources = ei[0][mask]
    attns   = avg_alpha[mask]

    if len(sources) == 0:
        print(f"Node {node_id} tidak punya tetangga.")
        return

    # Ambil top-k
    top_idx = np.argsort(attns)[::-1][:top_k]
    sources = sources[top_idx]
    attns   = attns[top_idx]

    colors = ['#FF6B6B' if df.iloc[s]['label'] == 1 else '#6BCB77' for s in sources]
    labels = [f"Node {s}\n{df.iloc[s]['sender'][:15]}" for s in sources]

    target_label = 'Spam' if df.iloc[node_id]['label'] == 1 else 'Ham'
    target_sender = df.iloc[node_id]['sender']

    plt.figure(figsize=(max(8, len(sources)), 4))
    bars = plt.bar(range(len(sources)), attns, color=colors, edgecolor='white', linewidth=0.5)
    plt.xticks(range(len(sources)), labels, rotation=30, ha='right', fontsize=8)
    plt.title(
        f'Attention ke Node {node_id} [{target_label}]\n'
        f'Sender: {target_sender[:40]}  |  GAT Layer {layer}',
        fontsize=11
    )
    plt.ylabel('Attention Weight')
    plt.ylim(0, max(attns) * 1.2)

    from matplotlib.patches import Patch
    plt.legend(handles=[
        Patch(color='#FF6B6B', label='Spam neighbor'),
        Patch(color='#6BCB77', label='Ham neighbor')
    ], loc='upper right')

    plt.tight_layout()
    plt.savefig(f'attention_node_{node_id}_layer{layer}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Pilih beberapa node untuk divisualisasikan
# Cari node yang punya tetangga (bukan isolated)
ei_np = data.edge_index.cpu().numpy()
nodes_with_neighbors = np.unique(ei_np[1])[:5]  # ambil 5 pertama

for node_id in nodes_with_neighbors[:3]:
    visualize_node_attention(df, edge_index_with_loops, alpha1, node_id=int(node_id), layer=1)

## 🔬 Cell 11: Error Analysis

In [ ]:
# Tambahkan hasil prediksi ke dataframe
model.eval()
with torch.no_grad():
    all_out   = model(data.x, data.edge_index)
    all_pred  = all_out.argmax(dim=1).cpu().numpy()
    all_probs = F.softmax(all_out, dim=1).cpu().numpy()

df_test = df.copy()
df_test['predicted']   = all_pred
df_test['prob_ham']    = all_probs[:, 0]
df_test['prob_spam']   = all_probs[:, 1]
df_test['is_correct']  = df_test['predicted'] == df_test['label']
df_test['in_test_set'] = False
df_test.loc[test_idx, 'in_test_set'] = True

test_df = df_test[df_test['in_test_set']]

# False Positives (Ham diprediksi Spam)
fp = test_df[(test_df['label'] == 0) & (test_df['predicted'] == 1)]
print(f"❌ FALSE POSITIVES (Ham → Spam): {len(fp)} email")
if len(fp) > 0:
    print(fp[['sender', 'subject', 'prob_spam']].head(3).to_string())

print()

# False Negatives (Spam lolos sebagai Ham)
fn = test_df[(test_df['label'] == 1) & (test_df['predicted'] == 0)]
print(f"⚠️  FALSE NEGATIVES (Spam → Ham): {len(fn)} email")
if len(fn) > 0:
    print(fn[['sender', 'subject', 'prob_ham']].head(3).to_string())

## 💾 Cell 12: Simpan Model & Artefak

In [ ]:
os.makedirs("saved_model", exist_ok=True)

# 1. Simpan weights model
torch.save(model.state_dict(), "saved_model/spamgat_weights.pt")

# 2. Simpan konfigurasi model
model_config = {
    'in_dim'     : 768,
    'hidden_dim' : 128,
    'num_classes': 2,
    'heads'      : 4,
    'dropout'    : 0.3,
    'bert_model' : BERT_MODEL,
    'max_length' : MAX_LENGTH,
}
import json
with open("saved_model/config.json", "w") as f:
    json.dump(model_config, f, indent=2)

# 3. Simpan graph data (node features + edge index)
torch.save({
    'x'          : node_features,
    'edge_index' : edge_index.cpu(),
    'y'          : labels,
}, "saved_model/graph_data.pt")

# 4. Simpan dataframe
df[['email_id', 'sender', 'subject', 'full_text', 'label', 'sender_domain']].to_csv(
    "saved_model/emails.csv", index=False
)

print("✅ Semua artefak berhasil disimpan di folder saved_model/:")
for f in os.listdir("saved_model"):
    size = os.path.getsize(f"saved_model/{f}") / 1024
    print(f"   {f:30s} {size:.1f} KB")

## 🚀 Cell 13: Inferensi Email Baru

In [ ]:
# ── Fungsi load model dari file ──
def load_model_for_inference():
    """Load semua komponen yang dibutuhkan untuk inferensi."""
    import json

    with open("saved_model/config.json") as f:
        config = json.load(f)

    # Load tokenizer & BERT
    tok = AutoTokenizer.from_pretrained(config['bert_model'])
    bert = AutoModel.from_pretrained(config['bert_model']).to(device)
    bert.eval()

    # Load GAT model
    gat = SpamGAT(
        in_dim=config['in_dim'],
        hidden_dim=config['hidden_dim'],
        num_classes=config['num_classes'],
        heads=config['heads'],
        dropout=config['dropout']
    ).to(device)
    gat.load_state_dict(torch.load("saved_model/spamgat_weights.pt", map_location=device))
    gat.eval()

    # Load graph data & dataframe
    graph = torch.load("saved_model/graph_data.pt", map_location=device)
    df_saved = pd.read_csv("saved_model/emails.csv")

    return tok, bert, gat, graph, df_saved, config


# ── Fungsi prediksi ──
def predict_email(sender, subject, body, tok, bert, gat, graph, df_saved, config):
    """
    Prediksi apakah sebuah email adalah spam atau ham.

    Returns dict dengan label, confidence, dan probabilitas.
    """
    # 1. Preprocess & embed email baru
    text = preprocess_text(subject + " [SEP] " + body)

    inputs = tok(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config['max_length'],
        padding="max_length"
    ).to(device)

    with torch.no_grad():
        out_bert = bert(**inputs)
    new_emb = out_bert.last_hidden_state[:, 0, :].squeeze(0).cpu()  # (768,)

    # 2. Sisipkan sebagai node baru di graf existing
    x_combined = torch.cat([graph['x'].cpu(), new_emb.unsqueeze(0)], dim=0)
    new_node_idx = x_combined.shape[0] - 1

    # 3. Hubungkan ke email dari sender yang sama
    same_sender_idx = df_saved[df_saved['sender'] == sender].index.tolist()

    new_edges = []
    for idx in same_sender_idx:
        new_edges.append([idx, new_node_idx])
        new_edges.append([new_node_idx, idx])

    if new_edges:
        new_edge_tensor = torch.tensor(new_edges, dtype=torch.long).T
        edge_combined   = torch.cat([graph['edge_index'].cpu(), new_edge_tensor], dim=1)
    else:
        # Sender baru → self-loop
        self_loop     = torch.tensor([[new_node_idx], [new_node_idx]], dtype=torch.long)
        edge_combined = torch.cat([graph['edge_index'].cpu(), self_loop], dim=1)

    # 4. Inferensi
    x_combined    = x_combined.to(device)
    edge_combined = edge_combined.to(device)

    with torch.no_grad():
        logits = gat(x_combined, edge_combined)
        probs  = F.softmax(logits[new_node_idx], dim=0)
        pred   = probs.argmax().item()

    return {
        'label'          : '🚨 SPAM' if pred == 1 else '✅ HAM',
        'prediction'     : pred,
        'confidence'     : f"{probs[pred].item() * 100:.1f}%",
        'prob_ham'       : f"{probs[0].item() * 100:.1f}%",
        'prob_spam'      : f"{probs[1].item() * 100:.1f}%",
        'sender_history' : f"{len(same_sender_idx)} email dari sender ini di training data"
    }


print("✅ Fungsi inferensi siap!")

In [ ]:
# Load model untuk inferensi
tok, bert_inf, gat_inf, graph, df_saved, config = load_model_for_inference()
print("✅ Model loaded untuk inferensi!")

# ── Test Case 1: Email Spam ──
result1 = predict_email(
    sender  = "promo@cheapdeal.com",
    subject = "You have won a prize!",
    body    = "Congratulations! You have been selected to receive a free iPhone. Click here to claim now!",
    tok=tok, bert=bert_inf, gat=gat_inf,
    graph=graph, df_saved=df_saved, config=config
)
print("\n" + "="*45)
print("📧 TEST 1: Suspicious email")
print("="*45)
for k, v in result1.items():
    print(f"  {k:20s}: {v}")

# ── Test Case 2: Email Normal ──
result2 = predict_email(
    sender  = "colleague@company.com",
    subject = "Meeting tomorrow at 10am",
    body    = "Hi, just a reminder about our weekly sync meeting tomorrow. Please bring the Q3 report.",
    tok=tok, bert=bert_inf, gat=gat_inf,
    graph=graph, df_saved=df_saved, config=config
)
print("\n" + "="*45)
print("📧 TEST 2: Normal email")
print("="*45)
for k, v in result2.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# ── Prediksi Batch dari List ──
def predict_batch(email_list, tok, bert, gat, graph, df_saved, config):
    """
    Prediksi banyak email sekaligus.
    email_list: list of dict dengan key 'sender', 'subject', 'body'
    """
    results = []
    for i, email in enumerate(email_list):
        result = predict_email(
            sender=email['sender'],
            subject=email.get('subject', ''),
            body=email.get('body', ''),
            tok=tok, bert=bert, gat=gat,
            graph=graph, df_saved=df_saved, config=config
        )
        results.append({
            'no'     : i + 1,
            'sender' : email['sender'],
            'subject': email.get('subject', '')[:40],
            'label'  : result['label'],
            'confidence': result['confidence']
        })
    return pd.DataFrame(results)


# Contoh batch prediction
test_emails = [
    {'sender': 'offer@promo-deal.net',
     'subject': 'Limited time offer!!!',
     'body': 'Buy now and get 90% discount! Limited stock available. Order immediately!'},

    {'sender': 'boss@mycompany.com',
     'subject': 'Project update',
     'body': 'Please send me the project status report by end of day. Thanks.'},

    {'sender': 'noreply@bank-alert.xyz',
     'subject': 'Urgent: Your account has been compromised',
     'body': 'Your account has suspicious activity. Verify your identity immediately or your account will be suspended.'},

    {'sender': 'friend@gmail.com',
     'subject': 'Weekend plans?',
     'body': 'Hey, are you free this weekend? We could grab coffee and catch up.'},
]

batch_results = predict_batch(
    test_emails, tok=tok, bert=bert_inf, gat=gat_inf,
    graph=graph, df_saved=df_saved, config=config
)

print("\n📊 Batch Prediction Results:")
print(batch_results.to_string(index=False))

## ☁️ Cell 14 (Opsional): Download Hasil ke Google Drive

In [ ]:
# Uncomment jika ingin simpan ke Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# shutil.copytree('saved_model', '/content/drive/MyDrive/SpamGAT/saved_model', dirs_exist_ok=True)
# shutil.copy('confusion_matrix.png',  '/content/drive/MyDrive/SpamGAT/')
# shutil.copy('training_history.png',  '/content/drive/MyDrive/SpamGAT/')

# print("✅ Semua file berhasil di-copy ke Google Drive!")

print("ℹ️  Uncomment cell ini jika ingin menyimpan ke Google Drive.")